# Exercise 7 — Hardware validation of two 4-bit registers (PYNQ-Z2)

**Target:** PYNQ-Z2 (Zynq-7020) · Vivado PS/PL design · AXI GPIO controlled from Python.

This notebook tests the **parallel 4-bit register**, **4-bit serial-in/parallel-out shift register (SIPO)** and **output multiplexer** implemented in `register_system.vhd`. It combines the validated hardware tests with an optional, notebook-hosted WebSocket server for the Flutter UI. No separate Python backend scripts are required.

### Hardware path

```text
PYNQ Python / ARM PS (Linux)
        | MMIO (M_AXI_GP0)
        v
AXI interconnect --> axi_gpio_0, base 0x41200000
                        | CH1: 9 output command bits
                        v
                register_command_adapter
                        | load, shift, mode, data, serial
               +--------+--------+
               v                 v
        parallel_register   shift_register
               +--------+--------+
                        | mode selects q[3:0]
                        v
                 AXI GPIO CH2 (4 inputs)
                        |
                        v
                 Python: q[3:0]
```

### Confirmed design parameters

| Item | Value |
| --- | --- |
| AXI GPIO base / MMIO range | `0x41200000` / `0x1000` |
| CH1 `DATA`, `TRI` | `0x00`, `0x04` |
| CH2 `DATA`, `TRI` | `0x08`, `0x0C` |
| Channel widths | 9 command outputs / 4 result inputs |
| Vivado `FCLK_CLK0` / timing target | **50 MHz** |
| Parallel output selection | `mode=0` |
| SIPO output selection | `mode=1` |
| Software reset command bit 8 | **Not wired** to `register_system.reset` |

> **Read carefully.** An access to an unresponsive PL AXI address previously made Linux, SSH and Jupyter unresponsive. Python `timeout`, a notebook interrupt or a `try/finally` block **cannot reliably recover** a stalled MMIO transaction. A power cycle may be required. Program the correct `.bit` and matching `.ltx` from Vivado first. The notebook **does not program the FPGA**. Run each cell in order and stop at the first unexpected result. **Do not** run all cells automatically on an unverified board.

Successful exercise-7 hardware results were previously observed on **2026-11-08**: parallel `0xA`, SIPO `0xB`. Those past results are *not* preloaded as notebook outputs; the tests below must run on the board to reproduce them.

## Step 1 — Software constants (no PL access)

Hardware changes required before running any MMIO cell:

1. Boot the PYNQ-Z2 and confirm Jupyter/SSH are responsive.
2. In Vivado Hardware Manager, **Program Device** with the exercise-7 `design_1_wrapper.bit` and matching `design_1_wrapper.ltx` (or otherwise establish that the exact same design is loaded).
3. Check that the Block Design provides `M_AXI_GP0` to AXI GPIO at `0x41200000` with the intended resets and clocks.
4. Confirm no conflicting notebook or program is driving this GPIO.

The explicit `HARDWARE_READY` flag is a deliberate guard, not automatic hardware detection. Set it to `True` **only after** the checks above.

In [1]:
import time
from datetime import datetime, timezone
from pynq import Clocks, MMIO

BASE_ADDR = 0x41200000
ADDR_RANGE = 0x1000
FCLK0_MHZ = 50.0

# AXI GPIO register offsets
GPIO_DATA  = 0x00
GPIO_TRI   = 0x04
GPIO2_DATA = 0x08
GPIO2_TRI  = 0x0C

# Command layout in register_command_adapter.vhd
DATA_MASK    = 0x00F
SERIAL_IN    = 0x010  # bit 4
MODE         = 0x020  # bit 5: 0=parallel, 1=shift
LOAD         = 0x040  # bit 6: rising-edge request
SHIFT        = 0x080  # bit 7: rising-edge request
RESET        = 0x100  # bit 8: NOT CONNECTED to register_system/reset
COMMAND_MASK = 0x1FF

# GPIO command pacing; not the FPGA clock period.
COMMAND_DELAY_SECONDS = 0.010

HARDWARE_READY = True  # MANUALLY change to True after verifying the loaded design

results = {}  # Populated by tests actually run in this kernel.
print(f"AXI GPIO: 0x{BASE_ADDR:08X}, range: 0x{ADDR_RANGE:X}")
print("Hardware confirmed by operator:", HARDWARE_READY)

AXI GPIO: 0x41200000, range: 0x1000
Hardware confirmed by operator: True


## Step 2 — Confirm hardware readiness and configure FCLK0

After a board restart, PYNQ previously reported **100 MHz** while this Vivado implementation was constrained for **50 MHz**. In the observed setup, `Clocks.fclk0_mhz = 50.0` changed the PS clock register, even though Linux debugfs continued to report a stale `100 MHz` value. Use the PYNQ clock reading as the immediate runtime check; the SLCR register can be examined separately if needed.

**Changing FCLK0 affects running PL logic and the ILA/debug hub.** Ensure that the programmed bitstream is the intended one before doing so. This step only touches clock configuration, not AXI GPIO.

In [2]:
if not HARDWARE_READY:
    raise RuntimeError(
        "Stop: verify the exercise-7 bitstream, address map and hardware state, "
        "then set HARDWARE_READY = True in Step 1 and rerun Step 1."
    )

before = Clocks.fclk0_mhz
print(f"FCLK0 before: {before:.2f} MHz")
if abs(before - FCLK0_MHZ) > 0.01:
    Clocks.fclk0_mhz = FCLK0_MHZ

after = Clocks.fclk0_mhz
print(f"FCLK0 after:  {after:.2f} MHz")
if abs(after - FCLK0_MHZ) > 0.01:
    raise RuntimeError("FCLK0 is not at the expected 50 MHz; do not access the AXI GPIO.")

clock_ok = True
print("Clock preparation: PASS")

FCLK0 before: 100.00 MHz
FCLK0 after:  50.00 MHz
Clock preparation: PASS


## Step 3 — Initialize MMIO (no read/write yet)

Creating `MMIO(...)` maps the address range. This **does not prove** that the slave responds. The first actual access happens in Step 4.

In [3]:
if not HARDWARE_READY or not globals().get("clock_ok", False):
    raise RuntimeError("Complete hardware and clock checks before creating MMIO.")

mmio = MMIO(BASE_ADDR, ADDR_RANGE)
print(f"MMIO mapped at 0x{BASE_ADDR:08X}")

MMIO mapped at 0x41200000


## Step 4 — First live AXI reads: GPIO direction diagnostics

**These are the first AXI GPIO transactions.** If the cell stops responding, do not re-execute it. Do not interpret the lack of a Python exception as proof that MMIO access cannot hang.

On this board, both `GPIO_TRI` reads returned `0xFFFFFFFF` even though the IP was configured with **`C_ALL_OUTPUTS=1` for CH1** and **`C_ALL_INPUTS_2=1` for CH2**, and both register tests later passed. Therefore **do not assert** that the returned TRI values must match the software-configurable direction bits. Record these readings as diagnostics; the Vivado IP configuration is authoritative for the fixed-direction ports.

Do **not** write the TRI registers as part of the functional test.

In [4]:
direction1 = mmio.read(GPIO_TRI)
direction2 = mmio.read(GPIO2_TRI)

print(f"CH1 TRI: 0x{direction1:08X}")
print(f"CH2 TRI: 0x{direction2:08X}")
print("Direction register values recorded (not a fixed-direction PASS/FAIL check).")
results["gpio_tri_read"] = "PASS"

CH1 TRI: 0xFFFFFFFF
CH2 TRI: 0xFFFFFFFF
Direction register values recorded (not a fixed-direction PASS/FAIL check).


## Step 5 — Command encoding and helper functions

```text
commands[8:0] = [RESET][SHIFT][LOAD][MODE][SERIAL_IN][D3][D2][D1][D0]
                    8      7     6    5      4         3..0

RESET=0x100  (NOT CONNECTED to register_system reset)
SHIFT=0x080  LOAD=0x040  MODE=0x020  SERIAL_IN=0x010
```

The VHDL command adapter synchronizes `LOAD` / `SHIFT` and converts a **0→1 transition** into a one-clock pulse. Each operation therefore writes: *prepare with request low → request high → request low*. The 10 ms pauses are for stable separate CPU writes, not a replacement for the HDL synchronizer.

A `finally` block tries to leave requests inactive if ordinary Python exceptions occur; it **cannot** run while a kernel-level MMIO access is stuck. Neither the `RESET` bit nor writing zero to CH1 is a general hardware reset. The SIPO test uses four shifts, which completely replace its previous four stored bits.

In [6]:
def write_commands(value: int) -> None:
    if not isinstance(value, int) or not 0 <= value <= COMMAND_MASK:
        raise ValueError(f"Command must be an integer in [0, 0x{COMMAND_MASK:X}]")
    mmio.write(GPIO_DATA, value)
    time.sleep(COMMAND_DELAY_SECONDS)


def read_register() -> int:
    return mmio.read(GPIO2_DATA) & DATA_MASK


def load_parallel(value: int) -> int:
    if not isinstance(value, int) or not 0 <= value <= DATA_MASK:
        raise ValueError("Parallel value must be a 4-bit integer")
    base = value  # mode=0, no LOAD/SHIFT assertion
    try:
        write_commands(base)
        write_commands(base | LOAD)
    finally:
        write_commands(base)  # release LOAD on ordinary Python errors
    return read_register()


def shift_bit(bit: int) -> int:
    if bit not in (0, 1):
        raise ValueError("Serial bit must be 0 or 1")
    base = MODE | (SERIAL_IN if bit else 0)
    try:
        write_commands(base)
        write_commands(base | SHIFT)
    finally:
        write_commands(base)  # release SHIFT on ordinary Python errors
    return read_register()


def select_mode(mode: int) -> int:
    if mode not in (0, 1):
        raise ValueError("mode must be 0 (parallel) or 1 (SIPO)")
    write_commands(MODE if mode else 0)
    return read_register()

print("Command helpers defined; no GPIO transactions performed in this cell.")

Command helpers defined; no GPIO transactions performed in this cell.


## Step 6 — Test parallel register (expected `0xA`)

This is the first **functional write** to the PL. It sends `0x00A → 0x04A → 0x00A` on CH1 and reads CH2. If a write stalls, do not restart the cell or issue more MMIO commands.

In [7]:
expected_parallel = 0xA
actual_parallel = load_parallel(expected_parallel)
results["parallel"] = "PASS" if actual_parallel == expected_parallel else "FAIL"

print("PARALLEL REGISTER")
print(f"Expected: 0x{expected_parallel:X} ({expected_parallel:04b})")
print(f"Actual:   0x{actual_parallel:X} ({actual_parallel:04b})")
print(f"Result:   {results['parallel']}")
assert actual_parallel == expected_parallel, "Parallel register mismatch"

PARALLEL REGISTER
Expected: 0xA (1010)
Actual:   0xA (1010)
Result:   PASS


## Step 7 — Test SIPO shift register (expected `0xB`)

Inject the bits `1, 0, 1, 1`. Intermediate states **may depend on the initial register content**, because software reset is not connected. After four shifts, however, all four old bits are replaced, so the **final result must be `1011`** regardless of initial state.

No software reset is attempted here.

In [9]:
sequence = [1, 0, 1, 1]
expected_shift = 0xB
shift_states = []

print("SIPO SHIFT REGISTER")
for bit in sequence:
    value = shift_bit(bit)
    shift_states.append(value)
    print(f"Bit injected: {bit} -> Register: {value:04b}")

actual_shift = shift_states[-1]
results["sipo"] = "PASS" if actual_shift == expected_shift else "FAIL"
print(f"Expected: 0x{expected_shift:X}")
print(f"Actual:   0x{actual_shift:X}")
print(f"Result:   {results['sipo']}")
assert actual_shift == expected_shift, "SIPO register mismatch"

SIPO SHIFT REGISTER
Bit injected: 1 -> Register: 0001
Bit injected: 0 -> Register: 0010
Bit injected: 1 -> Register: 0101
Bit injected: 1 -> Register: 1011
Expected: 0xB
Actual:   0xB
Result:   PASS


## Step 8 — Test output selection / independent storage

This step expects the **parallel register to contain `0xA`** and the **SIPO register to contain `0xB`**, as prepared by Steps 6 and 7. It checks that changing `mode` selects the output without issuing `LOAD` or `SHIFT` again.

To run this step independently, first rerun Steps 6 and 7.

In [10]:
if results.get("parallel") != "PASS" or results.get("sipo") != "PASS":
    raise RuntimeError("Run and pass parallel and SIPO tests before output-selection test")

parallel_selected = select_mode(0)
shift_selected = select_mode(1)
results["output_selection"] = (
    "PASS" if (parallel_selected, shift_selected) == (0xA, 0xB) else "FAIL"
)

print(f"Parallel selected: 0x{parallel_selected:X} ({parallel_selected:04b})")
print(f"SIPO selected:     0x{shift_selected:X} ({shift_selected:04b})")
print(f"Selection result:  {results['output_selection']}")
assert (parallel_selected, shift_selected) == (0xA, 0xB), "Output selection mismatch"

# Leave LOAD/SHIFT low and select parallel output.
write_commands(0)

Parallel selected: 0xA (1010)
SIPO selected:     0xB (1011)
Selection result:  PASS


## Step 9 — Summary and reproducibility record

This cell does not perform MMIO. It reports **only tests actually completed in the current notebook kernel**, not the successful historical results from 2026-11-08.

In [25]:
print("EXERCISE 7 — HARDWARE TEST SUMMARY")
print("UTC time:", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Board: PYNQ-Z2 / Zynq-7020")
print(f"AXI GPIO base: 0x{BASE_ADDR:08X}")
print(f"Target FCLK0: {FCLK0_MHZ:.1f} MHz")
for key, label in [
    ("gpio_tri_read", "CH1/CH2 TRI diagnostics"),
    ("parallel", "4-bit parallel register"),
    ("sipo", "4-bit SIPO shift register"),
    ("output_selection", "Output selection / storage"),
]:
    print(f"{label:32} {results.get(key, 'NOT RUN')}")

required = ("parallel", "sipo", "output_selection")
overall = "PASS" if all(results.get(x) == "PASS" for x in required) else "INCOMPLETE / FAIL"
print("Overall functional result:", overall)
print("Software reset: NOT TESTED (not connected in this design)")

## Troubleshooting notes and known limitations

### If MMIO hangs

- **Do not retry** the same cell, issue more MMIO accesses, or assume that notebook interruption can cancel a stalled hardware access.
- The problem may require physically rebooting the PYNQ-Z2. If Vivado Hardware Manager/JTAG interferes with reboot, disconnect the target before restarting.
- After reboot, confirm **the correct bitstream is programmed again**, then recheck FCLK0 (previously it returned to 100 MHz after boot).
- To investigate recurrent failures, arm a **System ILA** on the AXI interface *before* a controlled test: `ARVALID=1` for read requests, `AWVALID=1` for writes. Inspect `AR/ R` and `AW / W / B` handshake signals respectively. Capturing a later successful transaction does not prove why an earlier one hung.

### Confirmed vs unresolved (2026-11-08 investigation)

- **Confirmed:** `GPIO_TRI` reads completed, a standalone `GPIO_DATA` write returned and was observed with an AXI `BRESP=OKAY`, parallel loading `0xA` passed, and four SIPO shifts produced `0xB`.
- **Confirmed:** runtime FCLK0 was initially 100 MHz on boot, then set to 50 MHz in PYNQ. The SLCR `FPGA0_CLK_CTRL` register was observed as `0x00101400` while PYNQ reported 50 MHz; Linux debugfs still displayed 100 MHz.
- **Confirmed:** `proc_sys_reset.dcm_locked` was connected high, and the AXI active-low resets were captured deasserted.
- **Not established:** the **unique root cause** of prior freezes. Clock/reset changes and the final passing result were not isolated in controlled A/B experiments. Some earlier cells combined multiple MMIO operations, so the exact stalled access was not always known.
- **Not available:** command bit 8 (`RESET`) does not reach `register_system/reset`. The registers use an active-high asynchronous reset connected to `rst_ps7_0_50M/peripheral_reset`; adding software reset requires a deliberate hardware-design change.

### Version-controlled evidence

Keep the matching Vivado Block Design, RTL, bitstream generation instructions, timing constraints and screenshots/ILA captures together with the issue follow-up. **Do not commit a claim of a permanent MMIO hang fix without reproducible evidence.**

Reference: [fpga_lab issue #1](https://github.com/dexterBEN/fpga_lab/issues/1).

---

# Optional extension — Flutter WebSocket backend (same notebook)

The preceding Steps 1–9 remain the **hardware validation suite**. The cells below let Jupyter host a lightweight WebSocket server for Flutter without creating a `backend/` directory.

```text
Flutter UI -> ConnectionBloc / RegisterBloc -> WebSocketRegisterRepository
                         | WebSocket / JSON
                         v
PYNQ-Z2: this Jupyter notebook (WebSocket handler, shared lock)
                         |
                     PYNQ MMIO
                         |
                  AXI GPIO (CH1/CH2)
                         |
       parallel_register / shift_register (VHDL)
```

**Use only one owner of the GPIO at a time.** After starting this server, **do not rerun the notebook hardware test cells** while Flutter is connected. Stop the server first. A browser or WebSocket timeout **cannot rescue Linux from a blocked MMIO transaction**. No PL hardware reset is available through command bit 8.

**Run order:** finish hardware tests (or at least Steps 1–5), then run WebSocket Steps 10–13 in order. The Jupyter kernel must remain running while the server is in use. If you restart the kernel, verify/program PL and reinitialize FCLK0 before restarting the server.

**Network security:** default binding is `127.0.0.1`, accessible from the board only. To use Flutter on another machine in a trusted lab LAN, explicitly set `WS_HOST = "0.0.0.0"`. A random session token is required for *every* client request; use `ws://<pynq-ip>:8765` and provide the token in each JSON message. Do not expose this development server to the Internet. The token is not encrypted over plain WebSocket; use a protected LAN or a VPN/SSH tunnel for remote access.

## Step 10 — WebSocket dependency and configuration (no MMIO)

If `import websockets` fails in the **PYNQ Jupyter kernel**, install it in the environment used by that kernel, then restart/re-run the notebook. For example, in a notebook cell: `%pip install "websockets>=10,<17"` (requires package availability on the board). This notebook uses the compatible `websockets.legacy.server` API where present.

Keep `WS_HOST = "127.0.0.1"` until you are ready to connect over your local network. Choose a port not already used by Jupyter (the example uses 8765; Jupyter on 9090 is unrelated).

In [11]:
import asyncio
import json
import secrets
from collections import deque

try:
    import websockets
    try:
        from websockets.legacy.server import serve as ws_serve
    except ImportError:
        from websockets import serve as ws_serve
except ImportError as exc:
    raise RuntimeError(
        "Missing 'websockets' in the PYNQ Jupyter kernel. Install it there before continuing."
    ) from exc

WS_HOST = "0.0.0.0"  # Switch to "0.0.0.0" explicitly for a trusted LAN Flutter client.
WS_PORT = 8765
WS_TOKEN = secrets.token_urlsafe(24)  # New token on each execution of this cell.
MAX_MESSAGE_BYTES = 4096
HISTORY_LIMIT = 64

print(f"WebSocket library: {getattr(websockets, '__version__', 'unknown')}")
print(f"Bind: {WS_HOST}:{WS_PORT}")
print("Token (copy into the Flutter client; keep private):", WS_TOKEN)
print("No MMIO performed by this configuration cell.")

WebSocket library: 10.3
Bind: 0.0.0.0:8765
Token (copy into the Flutter client; keep private): ciTM-6Li5w2x7Y1XFQHbpBqaKTrXpIN0
No MMIO performed by this configuration cell.


## Step 11 — Protocol and hardware command handler (definitions only)

All hardware operations reuse the **same** helpers validated in Step 5: `load_parallel`, `shift_bit`, `select_mode`. The server protects **the entire write + read + restore sequence** with a shared `asyncio.Lock()`; this matters because the FPGA exposes only one multiplexed `q[3:0]` output.

**Client request examples** (every request has an `id` and a `token`):

```json
{"id":"r1","token":"<session-token>","action":"get_state"}
{"id":"r2","token":"<session-token>","action":"load_parallel","value":10}
{"id":"r3","token":"<session-token>","action":"shift","bit":1}
{"id":"r4","token":"<session-token>","action":"select_mode","mode":"shift"}
{"id":"r5","token":"<session-token>","action":"ping"}
```

**Responses:** one message per request with `type:"result"` (or `type:"pong"`), the matching `id`, and `state` when hardware was read. A mutation also broadcasts `type:"state_changed"` to all connected clients. `type:"error"` includes an `id` and a readable message. Connection `type:"hello"` does **not** claim hardware health.

`state.parallel.bits` and `state.shift.bits` are ordered **Q3 Q2 Q1 Q0**. All `parallel` and `shift` values are read **from the FPGA**, not simulated in Python. The `selected_mode` field is controller state, not an independently readable hardware register. The history tracks only commands run through this server and is cleared when this cell is re-executed. `shift_count` is a UI command count, not the 50 MHz hardware clock cycle count. No `reset` action is offered because software reset is not wired.

A `get_state` makes two mode-select writes and two reads, then restores the chosen mode. This snapshot is sequential, **not an atomic simultaneous hardware capture**.

In [12]:
# Per-kernel server state. Execute this cell once before starting the server.
# Do not re-execute it while the server is running.
ws_clients = set()
ws_server = None
ws_hardware_lock = asyncio.Lock()
ws_selected_mode = "parallel"
ws_shift_count = 0
ws_history = deque(maxlen=HISTORY_LIMIT)


def _ensure_bit(value, name):
    if type(value) is not int or value not in (0, 1):
        raise ValueError(f"{name} must be integer 0 or 1")
    return value


def _ensure_parallel(value):
    if type(value) is not int or not 0 <= value <= 15:
        raise ValueError("value must be an integer from 0 to 15")
    return value


def _ensure_mode(value):
    if value not in ("parallel", "shift"):
        raise ValueError("mode must be 'parallel' or 'shift'")
    return value


def _register_payload(value):
    return {"value": value, "bits": f"{value:04b}",
            "q0_to_q3": [(value >> i) & 1 for i in range(4)]}


def _hardware_snapshot():
    """Call ONLY while holding ws_hardware_lock. Reads both outputs sequentially."""
    desired_mode = ws_selected_mode
    try:
        parallel = select_mode(0)
        shift = select_mode(1)
    finally:
        # Restore the output mode. This is not a hardware reset.
        write_commands(MODE if desired_mode == "shift" else 0)
    return {
        "parallel": _register_payload(parallel),
        "shift": _register_payload(shift),
        "selected_mode": desired_mode,
        "shift_count": ws_shift_count,
        "history": list(ws_history),
        "fclk0_mhz": FCLK0_MHZ,  # Target configuration; not a live probe.
        "software_reset_available": False,
    }


def _process_hardware_request(request):
    """Synchronous MMIO calls; invoke only inside ws_hardware_lock."""
    global ws_selected_mode, ws_shift_count
    action = request["action"]
    if action == "get_state":
        return _hardware_snapshot(), False
    if action == "load_parallel":
        value = _ensure_parallel(request.get("value"))
        load_parallel(value)
        ws_selected_mode = "parallel"
        state = _hardware_snapshot()
        ws_history.append({"action": action, "value": value,
                           "result": state["parallel"]["value"]})
        state["history"] = list(ws_history)
        return state, True
    if action == "shift":
        bit = _ensure_bit(request.get("bit"), "bit")
        shift_bit(bit)
        ws_selected_mode = "shift"
        ws_shift_count += 1
        state = _hardware_snapshot()
        ws_history.append({"action": action, "bit": bit,
                           "cycle": ws_shift_count, "result": state["shift"]["value"]})
        state["history"] = list(ws_history)
        return state, True
    if action == "select_mode":
        mode = _ensure_mode(request.get("mode"))
        ws_selected_mode = mode
        state = _hardware_snapshot()
        return state, True
    raise ValueError("Unknown action. Allowed: ping, get_state, load_parallel, shift, select_mode")


def _parse_ws_request(raw):
    if isinstance(raw, bytes):
        raise ValueError("Only JSON text messages are supported")
    if len(raw.encode("utf-8")) > MAX_MESSAGE_BYTES:
        raise ValueError("Message too large")
    try:
        message = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON") from exc
    if not isinstance(message, dict):
        raise ValueError("Expected a JSON object")
    request_id = message.get("id")
    if type(request_id) is not str or not 1 <= len(request_id) <= 80:
        raise ValueError("id must be a nonempty string (up to 80 chars)")
    if not secrets.compare_digest(str(message.get("token", "")), WS_TOKEN):
        raise ValueError("Unauthorized: missing or invalid session token")
    if not isinstance(message.get("action"), str):
        raise ValueError("Missing action")
    return message


async def _broadcast_ws_state(state):
    if not ws_clients:
        return
    packet = json.dumps({"type": "state_changed", "state": state})
    targets = list(ws_clients)
    await asyncio.gather(*(client.send(packet) for client in targets),
                         return_exceptions=True)


async def _ws_client_handler(socket):
    ws_clients.add(socket)
    try:
        await socket.send(json.dumps({"type": "hello", "protocol": "fpga-exo7-v1",
                                      "hardware_verified": False,
                                      "note": "WebSocket open; send authenticated get_state to check PL"}))
        async for raw in socket:
            request_id = None
            try:
                # Pull id for error correlation even if other fields are invalid.
                try:
                    candidate = json.loads(raw) if isinstance(raw, str) else {}
                    if isinstance(candidate, dict) and isinstance(candidate.get("id"), str):
                        request_id = candidate["id"][:80]
                except (ValueError, TypeError):
                    pass
                request = _parse_ws_request(raw)
                request_id = request["id"]
                action = request["action"]
                if action == "ping":
                    await socket.send(json.dumps({"id": request_id, "type": "pong"}))
                    continue
                # IMPORTANT: all clients share the same lock for compound MMIO operations.
                async with ws_hardware_lock:
                    state, mutated = _process_hardware_request(request)
                await socket.send(json.dumps({"id": request_id, "type": "result", "state": state}))
                if mutated:
                    await _broadcast_ws_state(state)
            except (ValueError, KeyError, TypeError) as exc:
                await socket.send(json.dumps({"id": request_id, "type": "error", "message": str(exc)}))
            except Exception as exc:
                # An MMIO bus hang may never raise or reach this handler.
                print(f"WebSocket handler error: {type(exc).__name__}: {exc}")
                try:
                    await socket.send(json.dumps({"id": request_id, "type": "error",
                                                  "message": "Hardware operation failed; check server logs"}))
                except Exception:
                    break
    finally:
        ws_clients.discard(socket)


print("WebSocket protocol and hardware handlers defined (no MMIO yet).")

WebSocket protocol and hardware handlers defined (no MMIO yet).


## Step 12 — Start the server (non-blocking in Jupyter)

This is **not** a permanent system service. It runs only while this Jupyter kernel stays alive. This cell should return promptly; it does not loop forever. A Flutter client can connect to `ws://<PYNQ-IP>:8765` only if the bind address allows LAN access. Always set the token from Step 10 on every request.

Before running, make sure `HARDWARE_READY=True`, FCLK0 is still configured correctly, MMIO has been mapped, and the hardware tests are not currently running. The server does not perform MMIO until an authenticated client issues a hardware action (`get_state`, `load_parallel`, `shift`, `select_mode`).

In [13]:
if not HARDWARE_READY or not globals().get("clock_ok", False) or "mmio" not in globals():
    raise RuntimeError("Run hardware initialization Steps 1–5 before starting the WebSocket server")
if ws_server is not None:
    raise RuntimeError("Server already running: use the stop cell before starting again")
if abs(Clocks.fclk0_mhz - FCLK0_MHZ) > 0.01:
    raise RuntimeError("FCLK0 is no longer at 50 MHz; stop and check the PL configuration")

ws_server = await ws_serve(
    _ws_client_handler,
    WS_HOST,
    WS_PORT,
    max_size=MAX_MESSAGE_BYTES,
    ping_interval=20,
    ping_timeout=20,
)
print(f"WebSocket server RUNNING on ws://{WS_HOST}:{WS_PORT}")
print("The Jupyter kernel must stay alive. Do not rerun MMIO test cells while serving Flutter.")
print("This only confirms the socket is listening, not the health of the FPGA.")

WebSocket server RUNNING on ws://0.0.0.0:8765
The Jupyter kernel must stay alive. Do not rerun MMIO test cells while serving Flutter.
This only confirms the socket is listening, not the health of the FPGA.


## Step 13 — Stop the server safely

Execute this cell when you finish using Flutter, before re-running hardware tests or shutting down Jupyter. Closing the socket **does not cancel an MMIO operation already stalled inside Linux**. This cell does not reset the FPGA. You may then rerun Step 10 (new token), Step 11 and Step 12 for another session.

In [13]:
if ws_server is None:
    print("WebSocket server is already stopped.")
else:
    ws_server.close()
    await ws_server.wait_closed()
    ws_server = None
    print("WebSocket server stopped.")

if ws_clients:
    await asyncio.gather(*(client.close(code=1001, reason="Server stopped")
                           for client in list(ws_clients)), return_exceptions=True)
ws_clients.clear()

WebSocket server stopped.


## Flutter integration checklist

- `ConnectionBloc`: WebSocket state (`disconnected`, `connecting`, `connected`, `error`); **connected is not a hardware health check**.
- `RegisterBloc`: calls the shared repository, holds `parallel`, `shift`, `selected_mode`, serial bit, busy and error states, and produces animations only after a confirmed server snapshot.
- `WebSocketRegisterRepository`: one socket for both BLoCs, authenticated requests with unique IDs, response correlation and `state_changed` stream.
- Flutter renders bit positions **Q3 … Q0** from `bits`. Each request has one matching `result` or `error` response; `state_changed` is an additional event, not a reply.
- The UI **must not** present `Reset (UI)` as an FPGA hardware reset. It can clear UI animation/history only; the server's history is per notebook kernel.
- An `asyncio` lock serializes requests within this notebook server. This does **not** guard against another process opening `/dev/mem` or writing the same GPIO; do not run a second controller.
- `fclk0_mhz` in the payload is the configured target, **not** a live telemetry measurement.
- Historical SIPO values after four injected bits match `1011` regardless of initial register content; prior intermediate states may differ.

**Local smoke test** (run in a *different client*, not in the same MMIO validation cells): connect to the server, read the `hello` message, send authenticated `ping`, then `get_state`; only after hardware readiness is confirmed use `load_parallel` or `shift`. If an MMIO call causes SSH/Jupyter to hang, do not repeat it or rely on WebSocket ping timeouts to unlock the PS.


In [14]:
import asyncio
import json
import websockets

async def test_websocket():
    uri = "ws://127.0.0.1:8765"

    async with websockets.connect(uri) as ws:
        # Message envoyé automatiquement par le serveur
        hello = json.loads(await ws.recv())
        print("SERVER:", hello)

        # Test PING
        request = {
            "id": "test-001",
            "token": WS_TOKEN,
            "action": "ping"
        }

        await ws.send(json.dumps(request))

        response = json.loads(await ws.recv())
        print("PING RESPONSE:", response)

await test_websocket()

SERVER: {'type': 'hello', 'protocol': 'fpga-exo7-v1', 'hardware_verified': False, 'note': 'WebSocket open; send authenticated get_state to check PL'}
PING RESPONSE: {'id': 'test-001', 'type': 'pong'}


In [16]:
import json
import websockets
import uuid

async def send_command(action, **params):
    uri = "ws://127.0.0.1:8765"

    async with websockets.connect(uri) as ws:
        # Lire le message de bienvenue
        await ws.recv()

        request_id = str(uuid.uuid4())

        request = {
            "id": request_id,
            "token": WS_TOKEN,
            "action": action,
            **params,
        }

        await ws.send(json.dumps(request))

        # Ignorer les éventuels événements broadcast
        # pour récupérer la réponse correspondante.
        while True:
            response = json.loads(await ws.recv())

            if response.get("id") == request_id:
                print(json.dumps(response, indent=2))
                return response

In [17]:
# 1. Vérification réseau (sans MMIO)
await send_command("ping")

# 2. Lecture des deux registres (accès MMIO)
await send_command("get_state")

# 3. Chargement du registre parallèle avec 1010
await send_command("load_parallel", value=10)

# 4. Injection d'un bit série
await send_command("shift", bit=1)

# 5. Sélection du mode SIPO
await send_command("select_mode", mode="shift")

{
  "id": "f242591f-0863-4ca7-93fe-8ab4594c2bf0",
  "type": "pong"
}
{
  "id": "70738ac8-065a-4b4c-911d-f4c0bc9135fb",
  "type": "result",
  "state": {
    "parallel": {
      "value": 10,
      "bits": "1010",
      "q0_to_q3": [
        0,
        1,
        0,
        1
      ]
    },
    "shift": {
      "value": 11,
      "bits": "1011",
      "q0_to_q3": [
        1,
        1,
        0,
        1
      ]
    },
    "selected_mode": "parallel",
    "shift_count": 0,
    "history": [],
    "fclk0_mhz": 50.0,
    "software_reset_available": false
  }
}
{
  "id": "58a4621b-f2d1-4a2f-b75b-1b5d8d8a20d3",
  "type": "result",
  "state": {
    "parallel": {
      "value": 10,
      "bits": "1010",
      "q0_to_q3": [
        0,
        1,
        0,
        1
      ]
    },
    "shift": {
      "value": 11,
      "bits": "1011",
      "q0_to_q3": [
        1,
        1,
        0,
        1
      ]
    },
    "selected_mode": "parallel",
    "shift_count": 0,
    "history": [
      {
  

{'id': 'fb6f3b26-e286-4242-85f1-919fa25d5c24',
 'type': 'result',
 'state': {'parallel': {'value': 10, 'bits': '1010', 'q0_to_q3': [0, 1, 0, 1]},
  'shift': {'value': 7, 'bits': '0111', 'q0_to_q3': [1, 1, 1, 0]},
  'selected_mode': 'shift',
  'shift_count': 1,
  'history': [{'action': 'load_parallel', 'value': 10, 'result': 10},
   {'action': 'shift', 'bit': 1, 'cycle': 1, 'result': 7}],
  'fclk0_mhz': 50.0,
  'software_reset_available': False}}

In [15]:
print("WS_TOKEN:", repr(WS_TOKEN))

WS_TOKEN: 'ciTM-6Li5w2x7Y1XFQHbpBqaKTrXpIN0'
